In [1]:
!pip install -q pycolmap torchmetrics gsplat
!pip install -q git+https://github.com/compphoto/Intrinsic.git
!pip install -q opencv-python

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
import os

try:
    from google.colab import drive
    os.chdir('/content')
    drive.mount('/content/drive', force_remount=True)

    !pip install -q git+https://github.com/compphoto/Intrinsic.git
    !pip install opencv-python
    os.chdir('/content/drive/MyDrive/3dgs_decomposition')
except ModuleNotFoundError:
    pass  # running locally, already in project root

Mounted at /content/drive
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [3]:

import argparse, glob, os
import numpy as np, torch
import torch.nn.functional as F
from PIL import Image
from gsplat import rasterization
from gsplat.strategy import DefaultStrategy
from torchmetrics.functional import structural_similarity_index_measure as ssim_fn
import os

dev = "cuda"

def get_image_paths(root):
  img_dir = os.path.join(root, "images_8")

  paths = []
  for path in os.listdir(img_dir):
    if (path.endswith(".JPG") or path.endswith(".jpg")):
      paths.append(os.path.join(img_dir, path))

  return paths

import pycolmap

def getw2c(image):
    cam_from_world = image.cam_from_world
    if callable(cam_from_world):
        cam_from_world = cam_from_world()

    w2c = np.eye(4)
    w2c[:3, :4] = np.asarray(cam_from_world.matrix(), dtype=np.float64)
    return w2c

def get_cameras(root, paths):
  colmap_dir = os.path.join(root, "sparse/0")

  reconstruction = pycolmap.Reconstruction(colmap_dir)
  cameras = _as_dict(reconstruction.cameras)

  w2c_mats = []
  camera_ids = []
  by_name = {}

  images = _as_dict(reconstruction.images)
  image_ids = [int(image_id) for image_id in reconstruction.reg_image_ids()]
  imdata = {image_id: images[image_id] for image_id in image_ids}


  for image_id in imdata:
    im = imdata[image_id]
    w2c_mats.append(getw2c(im))
    camera_ids.append(im.camera_id)
    by_name[im.name] = (getw2c(im), int(im.camera_id))


  width, height = Image.open(paths[0]).size
  viewmats = []
  intrinsics = []
  for path in paths:
    name = os.path.basename(path)
    w2c, camera_id = by_name[name]
    cam = cameras[camera_id]
    K = np.asarray(cam.calibration_matrix(), dtype=np.float64).copy()
    K[0, :] *= width / cam.width
    K[1, :] *= height / cam.height

    intrinsics.append(K)
    viewmats.append(w2c)

  points = np.array([p.xyz for p in reconstruction.points3D.values()], dtype=np.float32)

  return (np.stack(viewmats), np.stack(intrinsics), (width, height), points,)


In [4]:
def _as_dict(map_like):
    return {int(k): v for k, v in map_like.items()}

In [5]:
import pycolmap

In [24]:
import os
import numpy as np
from PIL import Image
import torch
import torch.nn.functional as F


def _resize_to_hw(tensor: torch.Tensor, target_h: int, target_w: int) -> torch.Tensor:
  """Resize a tensor to (target_h, target_w) preserving trailing channels if present."""
  if tensor.ndim == 2:
    h, w = tensor.shape
    if (h, w) == (target_h, target_w):
      return tensor
    x = tensor.unsqueeze(0).unsqueeze(0)  # [1, 1, H, W]
    out = F.interpolate(x, size=(target_h, target_w), mode="bilinear", align_corners=False)
    return out.squeeze(0).squeeze(0)

  if tensor.ndim == 3:
    h, w, c = tensor.shape
    if (h, w) == (target_h, target_w):
      return tensor
    x = tensor.permute(2, 0, 1).unsqueeze(0)  # [1, C, H, W]
    out = F.interpolate(x, size=(target_h, target_w), mode="bilinear", align_corners=False)
    return out.squeeze(0).permute(1, 2, 0)

  raise ValueError(f"Expected 2D or 3D tensor, got shape {tuple(tensor.shape)}")


def get_views(paths, root, predictor, dataset, scene, dev="cuda"):
  images = []
  albedos = []
  depths = []

  for p in paths:
    file_name = os.path.splitext(os.path.basename(p))[0]
    print(file_name)

    with Image.open(p) as img_pil:
      img_rgb = img_pil.convert("RGB")
      width, height = img_rgb.size
      i = torch.tensor(np.array(img_rgb), dtype=torch.float32) / 255.0

    assert i.ndim == 3 and i.shape == (height, width, 3), (
        f"Expected ({height}, {width}, 3) for {p}, got {tuple(i.shape)}"
    )

    albedo_path = os.path.join(root, predictor, dataset, scene, "albedos", f"{file_name}.npy")
    albedo = torch.tensor(np.load(albedo_path), dtype=torch.float32)
    albedo = _resize_to_hw(albedo, height, width)

    depth_path = os.path.join(
        root, "depth_anything_v2", "depths", dataset, scene, f"{file_name}.png"
    )
    with Image.open(depth_path) as depth_pil:
      depth = torch.tensor(np.array(depth_pil, dtype=np.float32), dtype=torch.float32)

    depth = _resize_to_hw(depth, height, width)

    depth_npy_path = os.path.splitext(depth_path)[0] + ".npy"
    np.save(depth_npy_path, depth.detach().cpu().numpy())

    images.append(i)
    albedos.append(albedo)
    depths.append(depth.detach().clone())

  return (
      torch.stack(images, dim=0).to(dev),
      torch.stack(albedos, dim=0).to(dev),
      torch.stack(depths, dim=0).to(dev),
  )

In [7]:
def pearson_loss(d, m):
    d, m = d.flatten(), m.flatten()
    d = (d - d.mean()) / (d.std() + 1e-6); m = (m - m.mean()) / (m.std() + 1e-6)
    return 1 - (d * m).mean()

def photo_loss(pred, tgt, lam=0.2):
    if pred.ndim == 4 and pred.shape[0] == 1:
        pred = pred.squeeze(0)

    if tgt.ndim == 4 and tgt.shape[0] == 1:
        tgt = tgt.squeeze(0)
    l1 = (pred - tgt).abs().mean()
    pred = pred.permute(2, 0, 1).unsqueeze(0)
    tgt = tgt.permute(2, 0, 1).unsqueeze(0)

    s = 1 - ssim_fn(pred, tgt)

    return (1 - lam) * l1 + lam * s

def tv(x):
    return (x[1:] - x[:-1]).abs().mean() + (x[:, 1:] - x[:, :-1]).abs().mean()

In [8]:

learning_rates = {
    "means": 1.6e-4,
    "scales": 5e-3,
    "quats": 1e-3,
    "opacities": 5e-2,
    "colors": 1e-2,
}

def create_gaussians(means, scales, quats, opacities, colors, scene_scale=1.0):
    params = torch.nn.ParameterDict({
        "means": torch.nn.Parameter(means),
        "scales": torch.nn.Parameter(scales),
        "quats": torch.nn.Parameter(quats),
        "opacities": torch.nn.Parameter(opacities),
        "colors": torch.nn.Parameter(colors),
    })
    optimizers = {
        "means": torch.optim.Adam([params["means"]], lr=learning_rates["means"] * scene_scale),
        "scales": torch.optim.Adam([params["scales"]], lr=learning_rates["scales"]),
        "quats": torch.optim.Adam([params["quats"]], lr=learning_rates["quats"]),
        "opacities": torch.optim.Adam([params["opacities"]], lr=learning_rates["opacities"]),
        "colors": torch.optim.Adam([params["colors"]], lr=learning_rates["colors"]),
    }
    return params, optimizers

In [9]:
from scipy.spatial import cKDTree

def initialise_gaussians(raw_points, init_scale=1.0, init_opacity=0.1, scene_scale=1.0, dev=dev):
    pts_cpu = np.asarray(raw_points, dtype=np.float32)

    tree = cKDTree(pts_cpu)

    distances, _ = tree.query(pts_cpu, k=4)

    average_distance = np.maximum(distances[:, 1:].mean(axis=-1), 1e-4)

    points = torch.as_tensor(pts_cpu, dtype=torch.float32, device=dev)
    average_distance = torch.as_tensor(average_distance, dtype=torch.float32, device=dev)
    N = len(points)

    scales = torch.log(average_distance * init_scale).unsqueeze(-1).repeat(1, 3)
    quats = F.normalize(torch.rand((N, 4), device=dev), dim=-1)
    opacities = torch.logit(torch.full((N,), init_opacity, device=dev))
    colors = torch.zeros((N, 3), device=dev)


    return create_gaussians(points, scales, quats, opacities, colors, scene_scale=scene_scale)

In [10]:
def initialise_from_albedos(checkpoint, color_shape, initial_color=0.0):
  means = checkpoint["means"].to(dev).clone()
  scales = checkpoint["scales"].to(dev).clone()
  quats = checkpoint["quats"].to(dev).clone()
  opacities = checkpoint["opacities"].to(dev).clone()

  num_gaussians = len(means)
  colors = torch.full((num_gaussians, *color_shape), initial_color, device=dev)

  return create_gaussians(means, scales, quats, opacities, colors)


In [11]:
def render_gaussians(gaussians, viewmat, K, width, height, render_mode="RGB",
                     sh_degree=None, color_activation=None, verbose=False):
    means = gaussians["means"]
    quats = gaussians["quats"]
    scales = torch.exp(gaussians["scales"])
    opacities = torch.sigmoid(gaussians["opacities"])
    raw_colors = gaussians["colors"]
    colors = color_activation(raw_colors) if color_activation is not None else raw_colors

    if verbose:  # otherwise this prints every iteration
        print(f"[render_gaussians] N={means.shape[0]}, colors={tuple(colors.shape)}, "
              f"hw=({height},{width}), mode={render_mode}, sh_degree={sh_degree}")

    rendered_colors, rendered_alphas, info = rasterization(
        means=means, quats=quats, scales=scales, opacities=opacities, colors=colors,
        viewmats=viewmat.unsqueeze(0), Ks=K.unsqueeze(0),
        width=width, height=height,
        sh_degree=sh_degree, render_mode=render_mode,
        packed=False,   # strategy expects [C,N,2] means2d/radii
    )
    return rendered_colors[0], rendered_alphas[0], info

albdo = torch.sigmoid
shdo = F.softplus
resdo = lambda c: c

In [12]:
def step(loss, optimizers, strategy=None, state=None, params=None, step_idx=0, info=None):
    loss.backward()
    if strategy is not None:
      strategy.step_post_backward(
            params=params,
            optimizers=optimizers,
            state=state,
            step=step_idx,
            info=info,
        )
    for optimizer in optimizers.values():
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)


def save_checkpoint(params, checkpoint_directory, scene, predictor):
    os.makedirs(checkpoint_directory, exist_ok=True)
    path = os.path.join(checkpoint_directory, f"{scene}_{predictor}.pt")
    checkpoint = {name: tensor.detach().cpu() for name, tensor in params.items()}
    torch.save(checkpoint, path)
    print("Saved to: " + path)

def load_checkpoint(path):
  return {k: v.to(dev) for k, v in torch.load(path).items()}

In [13]:
import os
import numpy as np
from PIL import Image
import torch
import torch.nn.functional as F
from gsplat import rasterization


def _tensor_gib(t: torch.Tensor) -> float:
  return t.numel() * t.element_size() / (1024**3)


# def get_views(paths, root, predictor, dataset, scene, store_device="cpu"):
#   images = []
#   albedos = []
#   depths = []

#   for idx, p in enumerate(paths):
#     file_name = os.path.basename(p).split(".")[0]

#     with Image.open(p) as img:
#       width, height = img.size
#       rgb = np.asarray(img.convert("RGB"), dtype=np.float32) / 255.0

#     i = torch.from_numpy(rgb)
#     assert i.ndim == 3 and i.shape[2] == 3, (
#         f"Expected (H, W, 3) for {p}, got {tuple(i.shape)}"
#     )

#     albedo_path = os.path.join(
#         root, predictor, dataset, scene, "albedos", f"{file_name}.npy"
#     )
#     albedo = torch.from_numpy(np.asarray(np.load(albedo_path), dtype=np.float32))

#     depth_path = os.path.join(
#         root, "depth_anything_v2", "depths", dataset, scene, f"{file_name}.png"
#     )
#     with Image.open(depth_path) as d_img:
#       depth_np = np.asarray(d_img, dtype=np.float32)

#     depth = torch.from_numpy(depth_np)
#     depth_npy_path = os.path.splitext(depth_path)[0] + ".npy"
#     np.save(depth_npy_path, depth_np)

#     images.append(i.to(store_device))
#     albedos.append(albedo.to(store_device))
#     depths.append(depth.detach().clone().to(store_device))

#   stacked_images = torch.stack(images, dim=0)
#   stacked_albedos = torch.stack(albedos, dim=0)
#   stacked_depths = torch.stack(depths, dim=0)

#   return stacked_images, stacked_albedos, stacked_depths

In [ ]:
!cp -r /root/.cache/torch_extensions /content/drive/MyDrive/3dgs/torch_ext_cache


In [ ]:
!mkdir -p /root/.cache && cp -r /content/drive/MyDrive/3dgs/torch_ext_cache /root/.cache/torch_extensions


In [14]:
def to_dev(a):
    return torch.tensor(np.asarray(a), dtype=torch.float32, device=dev)

In [15]:
def fit_albedo(points, viewmats, intrinsics, width, height, images, albedos, depths, scene, predictor, ckpt_dir, iters=7000, lam_depth=0.1, snapshots=None):
    viewmats = torch.as_tensor(viewmats, dtype=torch.float32, device=dev)
    intrinsics = torch.as_tensor(intrinsics, dtype=torch.float32, device=dev)
    images = torch.as_tensor(images, dtype=torch.float32, device=dev)
    albedos = torch.as_tensor(albedos, dtype=torch.float32, device=dev)
    depths = torch.as_tensor(depths, dtype=torch.float32, device=dev)

    if albedos.max().item() > 1.5:
        albedos = albedos / 255.0

    params, optimizers = initialise_gaussians(points)
    strategy = DefaultStrategy(verbose=False)
    strategy.check_sanity(params, optimizers)
    state = strategy.initialize_state()

    for it in range(iters):
        v = np.random.randint(len(images))
        target_albedo = albedos[v:v+1].to(dev)
        target_depth = depths[v:v+1].to(dev)
        out, alphas, info = render_gaussians(params, viewmats[v], intrinsics[v], width, height, render_mode="RGB+ED", color_activation=albdo)

        out = out.squeeze(0)
        alphas = alphas.squeeze(0)

        pred = out[..., :3]
        depth = out[..., 3]

        strategy.step_pre_backward(params=params, optimizers=optimizers, state=state, step=it, info=info )

        if it == 0:
          print()

        albedo_loss = photo_loss(pred, target_albedo)

        depth_loss = pearson_loss(1.0 / depth.clamp(min=1e-3), target_depth)

        loss = albedo_loss + lam_depth * depth_loss

        step(loss, optimizers, strategy, state, params, it, info)

        del out, alphas, info
        del pred, depth, loss
        del albedo_loss, depth_loss
        del target_albedo, target_depth

    save_checkpoint(params, os.path.join(ckpt_dir, "albedo"), scene, predictor)

    return params

In [16]:
def freeze_albedo(viewmats, intrinsics, width, height, images, scene, predictor, ckpt_dir,
                iters=3000, lam_tv=0.05, snapshots=None):
    viewmats = torch.as_tensor(viewmats, dtype=torch.float32, device=dev)
    intrinsics = torch.as_tensor(intrinsics, dtype=torch.float32, device=dev)
    images = torch.as_tensor(images, dtype=torch.float32, device=dev)

    if (images.max().item() > 1.5):
      images = images/255.0
    path = os.path.join(ckpt_dir, "albedo", f"{scene}_{predictor}.pt")

    albedo_params = load_checkpoint(os.path.join(ckpt_dir, "albedo", f"{scene}_{predictor}.pt"))
    params, optimizers = initialise_from_albedos(albedo_params, (3,), initial_color=0.54)

    for it in range(iters):
      v = np.random.randint(len(images))
      target_image = images[v:v+1].to(dev)

      with torch.no_grad():
        albedo, albedo_alphas, _ = render_gaussians(albedo_params, viewmats[v], intrinsics[v], width, height, color_activation=albdo)
        albedo = albedo.squeeze(0)

      shading, shading_alphas, info = render_gaussians(params, viewmats[v], intrinsics[v], width, height, color_activation=shdo)
      shading = shading.squeeze(0)

      if it == 0:
            print(albedo.shape, shading.shape, target_image.shape)

      photo = photo_loss(albedo * shading, target_image)
      tv_loss = tv(shading)
      loss = photo + lam_tv * tv_loss
      step(loss, optimizers, step_idx=it)
      if it % 500 == 0:
          print(f"B {it}: loss {loss.item():.4f}")

      del albedo, albedo_alphas, shading, shading_alphas, info
      del photo, tv_loss, loss
      del target_image

    save_checkpoint(
        params,
        os.path.join(ckpt_dir, "shading"),
        scene,
        predictor
    )

    return params

In [17]:
def freeze_albedo_shading(viewmats, intrinsics, width, height, images, scene, predictor, ckpt_dir,
                iters=3000, lam_res=0.05, snapshots=None):
    viewmats = torch.as_tensor(viewmats, dtype=torch.float32, device=dev)
    intrinsics = torch.as_tensor(intrinsics, dtype=torch.float32, device=dev)
    images = torch.as_tensor(images, dtype=torch.float32, device=dev)

    if (images.max().item() > 1.5):
      images = images/255.0
    albedo_path = os.path.join(ckpt_dir, "albedo", f"{scene}_{predictor}.pt")
    shading_path = os.path.join(ckpt_dir, "shading", f"{scene}_{predictor}.pt")

    albedo_params = load_checkpoint(os.path.join(ckpt_dir, "albedo", f"{scene}_{predictor}.pt"))
    shading_params = load_checkpoint(os.path.join(ckpt_dir, "shading", f"{scene}_{predictor}.pt"))

    params, optimizers = initialise_from_albedos(albedo_params, (9,3), initial_color=0.0)

    for it in range(iters):
      v = np.random.randint(len(images))
      target_image = images[v:v+1].to(dev)

      with torch.no_grad():
        albedo, albedo_alphas, _ = render_gaussians(albedo_params, viewmats[v], intrinsics[v], width, height, color_activation=albdo)
        albedo = albedo.squeeze(0)
        shading, shading_alphas, info = render_gaussians(shading_params, viewmats[v], intrinsics[v], width, height, color_activation=shdo)
        shading = shading.squeeze(0)

      residuals, residuals_alphas, info = render_gaussians(params, viewmats[v], intrinsics[v], width, height, sh_degree=2, color_activation=resdo)
      residuals = residuals.squeeze(0) - SH_OFFSET * residuals_alphas.squeeze(0)


      if it == 0:
            print(albedo.shape, shading.shape, target_image.shape)

      photo = photo_loss(albedo * shading + residuals, target_image)

      res_loss = residuals.abs().mean()

      loss = photo + lam_res * res_loss

      step(loss, optimizers, step_idx=it)

      if it % 500 == 0:
          print(f"C {it}: loss {loss.item():.4f}")

      del albedo, albedo_alphas, shading, shading_alphas, residuals, residuals_alphas, info
      del photo, res_loss, loss
      del target_image


    save_checkpoint(
        params,
        os.path.join(ckpt_dir, "residual"),
        scene,
        predictor
    )

    return params

In [18]:
def check_priors_computed(ROOT, path, predictor, dataset, scene):
  image = os.path.splitext(os.path.basename(path))[0]
  if ((os.path.exists(os.path.join(ROOT, predictor, dataset, scene, "albedos", image + ".npy"))) and (os.path.exists(os.path.join(ROOT, "depth_anything_v2", "depths", dataset, scene, image + ".png")))):
        return True
  else:
    return False

In [19]:
def load(ROOT, predictor, dataset, scene):
  path = os.path.join(ROOT, "data", dataset, scene)
  print(path)
  all_paths = get_image_paths(path)[:5]

  paths_with_priors = [p for p in all_paths if check_priors_computed(ROOT, p, predictor, dataset, scene)]
  viewmats, intrinsics, (width, height), points = get_cameras(ROOT + "/data/" + dataset + "/" + scene, paths_with_priors)
  images, albedos, depths = get_views(paths_with_priors, ROOT, predictor, dataset, scene)

  res = {
      "root": ROOT,
      "paths": paths_with_priors,
      "viewmats": viewmats,
      "intrinsics": intrinsics,
      "width": width,
      "height": height,
      "points": points,
      "images": images,
      "albedos": albedos,
      "depths": depths
  }
  return res

In [20]:
def render_layers(albedo_parameters, shading_parameters, residuals_parameters, viewmat, K, width, height):
    albedo, _, _ = render_gaussians(albedo_parameters, viewmat, K, width, height, color_activation=albdo)
    shading, _, _ = render_gaussians(shading_parameters, viewmat, K, width, height, color_activation=shdo)
    residual, r_aux, _ = render_gaussians(residuals_parameters, viewmat, K, width, height, sh_degree=2, color_activation=resdo)
    return albedo, shading, residual - SH_OFFSET * r_aux

def save_preview(path, albedo_parameters, shading_parameters, residuals_parameters, viewmat, K, width, height, image):
    albedo, shading, residual = render_layers(albedo_parameters, shading_parameters, residuals_parameters, viewmat, K, width, height)
    tiles = [image.to(dev), albedo, shading, residual + 0.5, albedo * shading + residual]
    row = torch.cat([t.clamp(0, 1) for t in tiles], dim=1)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    Image.fromarray((row.cpu().numpy() * 255).astype(np.uint8)).save(path)

In [25]:
import tempfile, shutil

SH_OFFSET = 0.5

ROOT = "/content/drive/MyDrive/3dgs_decomposition"

def small_test(scene="room", predictor="colourful_iid", dataset="mipnerf360", iters=20):
    data = load(ROOT, predictor, dataset, scene)
    W, H = data["width"], data["height"]
    vm, K = data["viewmats"], data["intrinsics"]
    tmp = tempfile.mkdtemp()

    try:
        pts = data["points"][::10]
        fit_albedo(pts, vm, K, W, H, data["images"], data["albedos"], data["depths"],
                   scene, predictor, tmp, iters=iters)
        freeze_albedo(vm, K, W, H, data["images"], scene, predictor, tmp, iters=iters)
        freeze_albedo_shading(vm, K, W, H, data["images"], scene, predictor, tmp, iters=iters)

        A = load_checkpoint(f"{tmp}/albedo/{scene}_{predictor}.pt")
        S = load_checkpoint(f"{tmp}/shading/{scene}_{predictor}.pt")
        R = load_checkpoint(f"{tmp}/residual/{scene}_{predictor}.pt")

        a, s, r = render_layers(A, S, R, to_dev(vm)[0], to_dev(K)[0], W, H)
        full = a * s + r
        assert full.shape == (H, W, 3), f"bad render shape {tuple(full.shape)}"
        assert torch.isfinite(full).all(), "A*S+R contains NaN/inf"
        print(f"PASS: {len(A['means'])} Gaussians, A*S+R mean {full.mean():.3f}, "
              f"photo mean {data['images'][0].mean():.3f}")
    finally:
        shutil.rmtree(tmp, ignore_errors=True)

small_test()

/content/drive/MyDrive/3dgs_decomposition/data/mipnerf360/room
DSCF4667
DSCF4668
DSCF4669
DSCF4670
DSCF4671


Output()

gsplat: CUDA extension has been set up successfully in 959.44 seconds.

/usr/local/lib/python3.13/dist-packages/torchmetrics/utilities/prints.py:70: FutureWarning: Importing `spectral_angle_mapper` from `torchmetrics.functional` was deprecated and will be removed in 2.0. Import `spectral_angle_mapper` from `torchmetrics.image` instead.
  _future_warning(


Saved to: /tmp/tmpjdgk9okc/albedo/room_colourful_iid.pt
torch.Size([259, 389, 3]) torch.Size([259, 389, 3]) torch.Size([1, 259, 389, 3])
B 0: loss 0.2279
Saved to: /tmp/tmpjdgk9okc/shading/room_colourful_iid.pt
torch.Size([259, 389, 3]) torch.Size([259, 389, 3]) torch.Size([1, 259, 389, 3])
C 0: loss 0.1953
Saved to: /tmp/tmpjdgk9okc/residual/room_colourful_iid.pt
PASS: 11263 Gaussians, A*S+R mean 0.201, photo mean 0.229


In [26]:
!pip install torchmetrics lpips

In [27]:
import json
import os
import shutil
import tempfile
from argparse import Namespace
from pathlib import Path

import numpy as np
import pycolmap
import torch
import torch.nn.functional as F
from PIL import Image
from safetensors.torch import save_file

from eval.metrics import compute_lpips, compute_psnr, compute_ssim

PLY_GAUSSIAN_FIELDS = [
    "x", "y", "z",
    "nx", "ny", "nz",
    "f_dc_0", "f_dc_1", "f_dc_2",
    "opacity",
    "scale_0", "scale_1", "scale_2",
    "rot_0", "rot_1", "rot_2", "rot_3",
]

LAYER_SCALE_INVARIANT = {
    "albedo": True,
    "shading": True,
    "full": False,
}


def to_numpy(tensor):
    return tensor.detach().float().cpu().numpy()


def write_ply(path, params):
    means = to_numpy(params["means"])
    num_pts = len(means)
    colors = to_numpy(params["colors"]).reshape(num_pts, -1)[:, :3]
    opacities = to_numpy(params["opacities"]).reshape(num_pts, 1)
    scales = to_numpy(params["scales"]).reshape(num_pts, 3)
    quats = F.normalize(params["quats"].detach().float(), dim=-1).cpu().numpy()

    normals = np.zeros((num_pts, 3), dtype=np.float32)
    vertices = np.concatenate([means, normals, colors, opacities, scales, quats], axis=1).astype("<f4")

    header_lines = [
        "ply",
        "format binary_little_endian 1.0",
        f"element vertex {num_pts}",
        *(f"property float {field}" for field in PLY_GAUSSIAN_FIELDS),
        "end_header\n",
    ]

    out_path = os.fspath(path)
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    with open(out_path, "wb") as f:
        f.write("\n".join(header_lines).encode("ascii"))
        f.write(vertices.tobytes())


def generate_residual_safetensors(path, params):
    num_pts = params["means"].shape[0]
    colors = params["colors"].detach().float().cpu().reshape(num_pts, 9, 3)

    sh_rest = torch.zeros(num_pts, 15, 3)
    sh_rest[:, :8] = colors[:, 1:]

    tensors = {
        "_xyz": params["means"].detach().float().cpu(),
        "_features_dc": colors[:, :1],
        "_features_rest": sh_rest,
        "_scaling": params["scales"].detach().float().cpu(),
        "_rotation": F.normalize(params["quats"].detach().float(), dim=-1).cpu(),
        "_opacity": params["opacities"].detach().float().cpu().reshape(num_pts, 1),
    }

    out_path = os.fspath(path)
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    save_file({k: v.contiguous() for k, v in tensors.items()}, out_path)


def write_input_ply(scene_root, path):
    sparse_dir = os.path.join(scene_root, "sparse", "0")
    rec = pycolmap.Reconstruction(sparse_dir)
    points = list(rec.points3D.values())

    dtype = [
        ("x", "<f4"), ("y", "<f4"), ("z", "<f4"),
        ("nx", "<f4"), ("ny", "<f4"), ("nz", "<f4"),
        ("red", "u1"), ("green", "u1"), ("blue", "u1"),
    ]
    verts = np.zeros(len(points), dtype=dtype)
    xyz = np.array([p.xyz for p in points], dtype=np.float32)
    rgb = np.array([p.color for p in points], dtype=np.uint8)

    verts["x"], verts["y"], verts["z"] = xyz[:, 0], xyz[:, 1], xyz[:, 2]
    verts["red"], verts["green"], verts["blue"] = rgb[:, 0], rgb[:, 1], rgb[:, 2]

    header = [
        "ply",
        "format binary_little_endian 1.0",
        f"element vertex {len(verts)}",
        "property float x",
        "property float y",
        "property float z",
        "property float nx",
        "property float ny",
        "property float nz",
        "property uchar red",
        "property uchar green",
        "property uchar blue",
        "end_header\n",
    ]

    out_path = os.fspath(path)
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    with open(out_path, "wb") as f:
        f.write("\n".join(header).encode("ascii"))
        f.write(verts.tobytes())


def save_png(tensor, path):
    arr = (tensor.detach().clamp(0, 1) * 255).round().byte().cpu().numpy()
    Image.fromarray(arr).save(os.fspath(path))


def make_snapshot(out_dir, save_iters, writer_fn, filename):
    callbacks = {}
    for step in save_iters:
        target = os.path.join(out_dir, "point_cloud", f"iteration_{step}", filename)
        callbacks[step] = lambda params, path=target: writer_fn(path, params)
    return callbacks


@torch.no_grad()
def render_test_split(albedo_model, shading_model, residual_model,
                      viewmats, intrinsics, width, height, images, albedos, test_dir):
    for layer in LAYER_SCALE_INVARIANT:
        for subdir in ("renders", "gt"):
            os.makedirs(os.path.join(test_dir, layer, subdir), exist_ok=True)

    vm = to_dev(viewmats)
    K = to_dev(intrinsics)
    filenames = []

    for idx in range(len(vm)):
        name = f"{idx:05d}.png"
        filenames.append(name)

        photo = images[idx].to(dev)
        albedo_gt = albedos[idx].to(dev)
        shading_gt = (photo / albedo_gt.clamp(min=1e-2)).clamp(0, 1)

        a_pred, s_pred, r_pred = render_layers(
            albedo_model, shading_model, residual_model, vm[idx], K[idx], width, height
        )
        full_pred = (a_pred * s_pred + r_pred).clamp(0, 1)

        save_png(a_pred, os.path.join(test_dir, "albedo", "renders", name))
        save_png(albedo_gt, os.path.join(test_dir, "albedo", "gt", name))
        save_png(s_pred, os.path.join(test_dir, "shading", "renders", name))
        save_png(shading_gt, os.path.join(test_dir, "shading", "gt", name))
        save_png(full_pred, os.path.join(test_dir, "full", "renders", name))
        save_png(photo, os.path.join(test_dir, "full", "gt", name))

    per_view = {layer: {"PSNR": {}, "SSIM": {}, "LPIPS": {}} for layer in LAYER_SCALE_INVARIANT}
    for layer, scale_invariant in LAYER_SCALE_INVARIANT.items():
        for name in filenames:
            pred_path = os.path.join(test_dir, layer, "renders", name)
            gt_path = os.path.join(test_dir, layer, "gt", name)
            per_view[layer]["PSNR"][name] = compute_psnr(pred_path, gt_path, scale_invariant)
            per_view[layer]["SSIM"][name] = compute_ssim(pred_path, gt_path, scale_invariant)
            per_view[layer]["LPIPS"][name] = compute_lpips(pred_path, gt_path, scale_invariant)

    results = {
        layer: {metric: float(np.mean(list(scores.values()))) for metric, scores in metrics.items()}
        for layer, metrics in per_view.items()
    }
    return results, per_view

Setting up [LPIPS] perceptual loss: trunk [vgg], v[0.1], spatial [off]


/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:207: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/torchvision/models/_utils.py:222: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:07<00:00, 71.7MB/s]


Loading model from: /usr/local/lib/python3.13/dist-packages/lpips/weights/v0.1/vgg.pth


In [28]:
def run_gsplat_pipeline(root, dataset, scene, predictor, output_dir=None, iterations=10_000,
        snapshot_iters=(7_000,), eval=True, llffhold=8, max_views=None, seed=0):

    data_root = os.fspath(root)

    scene_root = os.path.join(data_root, "data", dataset, scene)

    if output_dir is None:
        output_dir = os.path.join(data_root, "gsplat_outputs", predictor, dataset, scene)
    output_dir = os.fspath(output_dir)
    os.makedirs(output_dir, exist_ok=True)

    np.random.seed(seed)
    torch.manual_seed(seed)

    all_paths = sorted(get_image_paths(scene_root))
    if max_views:
        all_paths = all_paths[:max_views]

    is_test = [eval and (i % llffhold == 0) for i in range(len(all_paths))]
    has_priors = [
        check_priors_computed(data_root, p, predictor, dataset, scene)
        for p in all_paths
    ]

    train_paths = [p for p, test, ok in zip(all_paths, is_test, has_priors) if ok and not test]
    test_paths = [p for p, test, ok in zip(all_paths, is_test, has_priors) if ok and test]

    paths = train_paths + test_paths
    n_train = len(train_paths)

    skipped = len(all_paths) - len(paths)
    print(f"{n_train} train and {len(test_paths)} test views - skipped: {skipped}")

    viewmats, intrinsics, (width, height), points = get_cameras(scene_root, paths)
    images, albedos, depths = get_views(paths, data_root, predictor, dataset, scene)

    if images.max().item() > 1.5:
        images = images / 255.0
    if albedos.max().item() > 1.5:
        albedos = albedos / 255.0

    train_slice = slice(0, n_train)
    test_slice = slice(n_train, len(paths))

    save_iters = sorted({step for step in snapshot_iters if step <= iterations} | {iterations})
    tmp_dir = tempfile.mkdtemp()

    fit_albedo(
        points, viewmats[train_slice], intrinsics[train_slice], width, height,
        images[train_slice], albedos[train_slice], depths[train_slice],
        scene, predictor, tmp_dir, iters=iterations,
        snapshots=make_snapshot(output_dir, save_iters, write_ply, "albedo.ply"),
    )

    freeze_albedo(
        viewmats[train_slice], intrinsics[train_slice], width, height, images[train_slice],
        scene, predictor, tmp_dir, iters=iterations,
        snapshots=make_snapshot(output_dir, save_iters, write_ply, "shading.ply"),
    )

    albedo_model = load_checkpoint(os.path.join(tmp_dir, "albedo", f"{scene}_{predictor}.pt"))
    shading_model = load_checkpoint(os.path.join(tmp_dir, "shading", f"{scene}_{predictor}.pt"))
    residual_model = load_checkpoint(os.path.join(tmp_dir, "residual", f"{scene}_{predictor}.pt"))

    write_input_ply(scene_root, os.path.join(output_dir, "input.ply"))

    cfg = Namespace(
        source_path=scene_root,
        model_path=output_dir,
        images="images_8",
        eval=eval,
        llffhold=llffhold,
        predictor=predictor,
        iterations=iterations,
        albedo_iterations=iterations,
        shading_iterations=iterations,
        residual_iterations=iterations,
        save_iterations=save_iters,
        seed=seed,
    )

    with open(os.path.join(output_dir, "cfg_args"), "w", encoding="utf-8") as f:
        f.write(str(cfg))

    if eval and test_paths:
        tag = f"ours_{iterations}"
        results, per_view = render_test_split(
            albedo_model,
            shading_model,
            residual_model,
            viewmats[test_slice],
            intrinsics[test_slice],
            width,
            height,
            images[test_slice],
            albedos[test_slice],
            os.path.join(out_dir, "test", tag),
        )
        with open(os.path.join(output_dir, "results.json"), "w", encoding="utf-8") as f:
            json.dump({tag: results}, f, indent=2)
        with open(os.path.join(output_dir, "per_view.json"), "w", encoding="utf-8") as f:
            json.dump({tag: per_view}, f, indent=2)
        print(f"[run] test metrics ({tag}): {results}")

    print(f"Run completed: {output_dir}")
    return output_dir

In [ ]:
ROOT = "/content/drive/MyDrive/3dgs_decomposition"

run_gsplat_pipeline(ROOT, "mipnerf360", "room", "colourful_iid")

272 train and 39 test views - skipped: 0
DSCF4668
DSCF4669
DSCF4670
DSCF4671
DSCF4672
DSCF4673
DSCF4674
DSCF4676
DSCF4677
DSCF4678
DSCF4679
DSCF4680
DSCF4681
DSCF4682
DSCF4684
DSCF4685
DSCF4686
DSCF4687
DSCF4688
DSCF4689
DSCF4690
DSCF4692
DSCF4693
DSCF4694
DSCF4695
DSCF4696
DSCF4697
DSCF4698
DSCF4700
DSCF4701
DSCF4702
DSCF4703
DSCF4704
DSCF4705
DSCF4706
DSCF4708
DSCF4709
DSCF4710
DSCF4711
DSCF4712
DSCF4713
DSCF4714
DSCF4716
DSCF4717
DSCF4718
DSCF4719
DSCF4720
DSCF4721
DSCF4722
DSCF4724
DSCF4725
DSCF4726
DSCF4727
DSCF4728
DSCF4729
DSCF4730
DSCF4732
DSCF4733
DSCF4734
DSCF4735
DSCF4736
DSCF4737
DSCF4738
DSCF4740
DSCF4741
DSCF4742
DSCF4743
DSCF4744
DSCF4745
DSCF4746
DSCF4748
DSCF4749
DSCF4750
DSCF4751
DSCF4752
DSCF4753
DSCF4754
DSCF4756
DSCF4757
DSCF4758
DSCF4759
DSCF4760
DSCF4761
DSCF4762
DSCF4764
DSCF4765
DSCF4766
DSCF4767
DSCF4768
DSCF4769
DSCF4770
DSCF4772
DSCF4773
DSCF4774
DSCF4775
DSCF4776
DSCF4777
DSCF4778
DSCF4780
DSCF4781
DSCF4782
DSCF4783
DSCF4784
DSCF4785
DSCF4786
DSCF4788
DSCF4